In [ ]:
import numpy as np
import pandas as pd
import os
import zipfile
from pathlib import Path


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import zipfile
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

# === CONFIGURATION ===
# --- File Paths ---
ZIP_PATH = "/content/drive/MyDrive/market_data/Copy of Copy of SOUN.zip"
EXTRACT_DIR = "/content/ticker_data/SOUN"
OUTPUT_CSV = "/content/SOUN_final_dataset_v3.csv"

# --- Simulation Parameters ---
# The number of levels of the order book to consider.
TOP_LEVELS = 10
# The total number of simulated trades to generate for the final dataset.
NUM_SIMULATED_TRADES = 20000
# The proportion of small, medium, and large trades to simulate.
TRADE_SIZE_PROPORTIONS = (0.4, 0.35, 0.25)


# === STEP 1: UNZIP ALL CSVs ===
os.makedirs(EXTRACT_DIR, exist_ok=True)
print(f"Extracting files to {EXTRACT_DIR}...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
print(f"✅ Extraction complete.")


# === HELPER FUNCTIONS ===

def preprocess_df(df):
    """
    Cleans the initial raw DataFrame from a single CSV.
    """
    df['ts_event'] = pd.to_datetime(df['ts_event'], format='mixed', errors='coerce')
    df.dropna(subset=['ts_event'], inplace=True)
    df['minute'] = df['ts_event'].dt.floor('min')
    columns_to_drop = [
        'ts_event.1', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side',
        'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence'
    ]
    df.drop(columns=[c for c in columns_to_drop if c in df.columns], inplace=True)
    return df

def get_minute_snapshots(df):
    """
    Aggregates tick-level data into one-minute snapshots by taking the mean.
    """
    feature_cols = [f'{side}_{field}_0{i}' for i in range(TOP_LEVELS)
                      for side in ['bid', 'ask'] for field in ['px', 'sz', 'ct']]
    feature_cols = [col for col in feature_cols if col in df.columns]
    grouped = df.groupby(['minute', 'symbol'], sort=False)[feature_cols].mean().reset_index()
    return grouped

def compute_features(df):
    """
    Engineers a rich set of features from the minute-level order book snapshots.
    """
    df = df.copy()
    df['mid_price'] = (df['bid_px_00'] + df['ask_px_00']) / 2
    df['spread'] = df['ask_px_00'] - df['bid_px_00']
    df['imbalance'] = df['bid_sz_00'] / (df['bid_sz_00'] + df['ask_sz_00'] + 1e-6)

    df['cum_bid_size'] = sum([df.get(f'bid_sz_0{i}', 0) for i in range(TOP_LEVELS)])
    df['cum_ask_size'] = sum([df.get(f'ask_sz_0{i}', 0) for i in range(TOP_LEVELS)])
    df['cum_bid_count'] = sum([df.get(f'bid_ct_0{i}', 0) for i in range(TOP_LEVELS)])
    df['cum_ask_count'] = sum([df.get(f'ask_ct_0{i}', 0) for i in range(TOP_LEVELS)])

    df['bid_width'] = df['bid_px_00'] - df.get('bid_px_09', df['bid_px_00'])
    df['ask_width'] = df.get('ask_px_09', df['ask_px_00']) - df['ask_px_00']
    df['bid_price_slope'] = df['bid_width'] / (df['cum_bid_size'] + 1e-6)
    df['ask_price_slope'] = df['ask_width'] / (df['cum_ask_size'] + 1e-6)

    df['size_asymmetry'] = df['cum_bid_size'] - df['cum_ask_size']
    df['count_asymmetry'] = df['cum_bid_count'] - df['cum_ask_count']

    df['minute_of_day'] = df['minute'].dt.hour * 60 + df['minute'].dt.minute
    df['sin_time'] = np.sin(2 * np.pi * df['minute_of_day'] / 390)
    df['cos_time'] = np.cos(2 * np.pi * df['minute_of_day'] / 390)
    return df

def simulate_slippage(row, x):
    """
    Simulates a market buy order of size 'x' against a given order book snapshot.
    """
    total_cost = 0
    shares_left = x
    for i in range(TOP_LEVELS):
        price = row.get(f'ask_px_0{i}', np.nan)
        size = row.get(f'ask_sz_0{i}', 0)
        if np.isnan(price) or shares_left <= 0:
            break
        trade_size = min(size, shares_left)
        total_cost += trade_size * price
        shares_left -= trade_size
    if shares_left > 0:
        return np.nan
    avg_exec_price = total_cost / x
    return avg_exec_price - row['mid_price']

def generate_simulation_chunk(features_df, num_samples_to_generate):
    """
    Generates a chunk of simulated data from a features DataFrame.
    """
    # Ensure we don't try to sample more than available, unless replacing.
    num_samples = min(len(features_df), num_samples_to_generate)

    if num_samples == 0:
        return pd.DataFrame()

    n_small = int(num_samples * TRADE_SIZE_PROPORTIONS[0])
    n_medium = int(num_samples * TRADE_SIZE_PROPORTIONS[1])
    n_large = num_samples - n_small - n_medium

    sampled = features_df.sample(n=num_samples, replace=True, random_state=42).copy()

    x_small = np.random.randint(50, 300, size=n_small)
    x_medium = np.random.randint(300, 800, size=n_medium)
    x_large = np.random.randint(800, 1500, size=n_large)

    x_all = np.concatenate([x_small, x_medium, x_large])
    np.random.shuffle(x_all)

    sampled['x'] = x_all
    sampled['slippage'] = sampled.apply(lambda row: simulate_slippage(row, row['x']), axis=1)
    return sampled.dropna(subset=['slippage'])


# === MAIN PROCESSING SCRIPT (MEMORY EFFICIENT) ===
csv_files = glob.glob(os.path.join(EXTRACT_DIR, "*.csv"))
print(f"📁 Found {len(csv_files)} CSV files to process.")

if not csv_files:
    print("No CSV files found. Exiting.")
else:
    # Calculate how many samples to generate from each file to reach the target
    num_samples_per_file = max(1, NUM_SIMULATED_TRADES // len(csv_files))
    print(f"Aiming for ~{num_samples_per_file} samples per file.")

    all_simulations = []
    for fpath in tqdm(csv_files, desc="Processing files and simulating trades"):
        try:
            df = pd.read_csv(fpath, low_memory=False)
            df_processed = preprocess_df(df)
            df_snapshots = get_minute_snapshots(df_processed)
            df_features = compute_features(df_snapshots)

            if df_features.empty:
                continue

            # Generate a small chunk of simulated data from this file's features
            simulated_chunk = generate_simulation_chunk(df_features, num_samples_to_generate=num_samples_per_file)
            all_simulations.append(simulated_chunk)

        except Exception as e:
            print(f"❌ Failed on {fpath}: {e}")
            continue

    if not all_simulations:
        print("No simulations could be generated. Exiting.")
    else:
        # Combine all the small simulated chunks into one final DataFrame
        final_df = pd.concat(all_simulations, ignore_index=True)
        print(f"\n✅ Combined all simulations into {len(final_df)} total rows.")

        # Sample down to the exact target number if we have more than enough
        if len(final_df) > NUM_SIMULATED_TRADES:
            final_df = final_df.sample(n=NUM_SIMULATED_TRADES, random_state=42)

        # Select and order final columns
        final_columns = [
            'minute', 'symbol', 'x', 'mid_price', 'spread', 'imbalance',
            'cum_bid_size', 'cum_ask_size', 'cum_bid_count', 'cum_ask_count',
            'bid_width', 'ask_width', 'bid_price_slope', 'ask_price_slope',
            'size_asymmetry', 'count_asymmetry', 'sin_time', 'cos_time',
            'slippage'
        ]
        final_df = final_df[final_columns]

        # Save the final dataset
        final_df.to_csv(OUTPUT_CSV, index=False)
        print(f"📊 Final dataset with {len(final_df)} simulated trades saved to: {OUTPUT_CSV}")



Extracting files to /content/ticker_data/SOUN...
✅ Extraction complete.
📁 Found 21 CSV files to process.
Aiming for ~952 samples per file.


Processing files and simulating trades: 100%|██████████| 21/21 [01:51<00:00,  5.30s/it]



✅ Combined all simulations into 8190 total rows.
📊 Final dataset with 8190 simulated trades saved to: /content/SOUN_final_dataset_v3.csv


In [ ]:
df = pd.read_csv('/content/SOUN_final_dataset_v3.csv')

In [ ]:
df.shape

(8190, 19)

In [ ]:
df.head()

,minute,symbol,x,mid_price,spread,imbalance,cum_bid_size,cum_ask_size,cum_bid_count,cum_ask_count,bid_width,ask_width,bid_price_slope,ask_price_slope,size_asymmetry,count_asymmetry,sin_time,cos_time,slippage
0,2025-04-17 15:12:00+00:00,SOUN,132,7.720460,0.010077,0.565144,28904.728900,29058.391304,87.907928,67.271100,0.09,0.09,0.000003,0.000003,-153.662404,20.636829,0.849468,-0.527640,0.005038
1,2025-04-17 19:18:00+00:00,SOUN,1261,7.875076,0.011152,0.417547,47506.423913,49130.373913,111.426087,89.417391,0.09,0.09,0.000002,0.000002,-1623.950000,22.008696,-0.192127,0.981370,0.005576
2,2025-04-17 18:00:00+00:00,SOUN,1301,7.911313,0.010369,0.739309,34413.391705,50819.976959,81.281106,113.202765,0.09,0.09,0.000003,0.000002,-16406.585253,-31.921659,-0.992709,0.120537,0.005184
3,2025-04-17 15:16:00+00:00,SOUN,1052,7.700539,0.011198,0.453485,26448.724551,33244.649701,80.110778,97.305389,0.09,0.09,0.000003,0.000003,-6795.925150,-17.194611,0.813726,-0.581249,0.005599
4,2025-04-17 14:41:00+00:00,SOUN,195,7.637600,0.011404,0.540047,29270.796791,26491.887701,103.883690,66.498663,0.09,0.09,0.000003,0.000003,2778.909091,37.385027,0.998411,-0.056358,0.005702
